# Alpha Vantage Exploration for Bronze Layer

This notebook explores the Alpha Vantage datasets that are the most useful for the financial data platform using only endpoints available in the free plan.

Project goal for this notebook:
- explore stock daily prices
- inspect company overview and fundamentals
- inspect FX daily data
- inspect one macroeconomic indicator
- design bronze payloads and file layout for S3


## Step 1. What we want from Alpha Vantage

Alpha Vantage is a strong complement to Yahoo Finance because it gives us more structured datasets for:
- fundamentals
- FX
- macroeconomic indicators
- daily stock prices

Recommended first datasets for this project:
- `TIME_SERIES_DAILY`
- `OVERVIEW`
- `INCOME_STATEMENT`
- `BALANCE_SHEET`
- `FX_DAILY`
- `REAL_GDP`

Important note:
- as of March 23, 2026, `TIME_SERIES_DAILY_ADJUSTED` is documented by Alpha Vantage as a premium endpoint
- for the free plan, we will use `TIME_SERIES_DAILY` here and rely on Yahoo Finance for split and dividend exploration

Official documentation:
- https://www.alphavantage.co/documentation/

## Step 2. Configure the API key

Put your Alpha Vantage key in a `.env` file at the project root.

Example:

```env
ALPHA_VANTAGE_API_KEY=your_key_here
```

If the key is missing, the notebook will raise an error so you notice it early.

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import time

import pandas as pd
import requests
from dotenv import load_dotenv

In [4]:
load_dotenv()

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

API_KEY = os.getenv("ALPHA_VANTAGE_API_KEY")
BASE_URL = "https://www.alphavantage.co/query"

if not API_KEY:
    raise ValueError("ALPHA_VANTAGE_API_KEY not found. Add it to your .env file before running this notebook.")

stock_symbol = "IBM"
fx_from = "USD"
fx_to = "BRL"

print("Stock symbol:", stock_symbol)
print("FX pair:", f"{fx_from}/{fx_to}")

Stock symbol: IBM
FX pair: USD/BRL


## Step 3. Create a reusable request helper

This helper keeps the exploration notebook clean and gives us a structure that later can become a Python client in `src/clients/alpha_vantage.py`.

In [5]:
def call_alpha_vantage(**params):
    request_params = {**params, "apikey": API_KEY}
    response = requests.get(BASE_URL, params=request_params, timeout=30)
    response.raise_for_status()

    if "application/json" in response.headers.get("Content-Type", ""):
        payload = response.json()
    else:
        payload = response.text

    return {
        "request_url": response.url.replace(API_KEY, "***REDACTED***"),
        "status_code": response.status_code,
        "payload": payload,
    }


def inspect_alpha_payload(payload):
    if isinstance(payload, dict):
        for key in ["Information", "Note", "Error Message"]:
            if key in payload:
                print(f"{key}: {payload[key]}")
    else:
        print("Non-JSON payload received")

## Step 4. Explore free daily stock prices

We will use `TIME_SERIES_DAILY`, which is part of the free documentation.

This dataset gives us:
- open
- high
- low
- close
- volume

This is enough for bronze ingestion and for many silver and gold transformations. For dividend and split data, Yahoo Finance is still the better free source in this project.

In [ ]:
daily_response = call_alpha_vantage(
    function="TIME_SERIES_DAILY",
    symbol=stock_symbol,
    outputsize="compact",
)

print("Status code:", daily_response["status_code"])
print("Request URL:", daily_response["request_url"])
inspect_alpha_payload(daily_response["payload"])

if isinstance(daily_response["payload"], dict):
    print(list(daily_response["payload"].keys())[:10])

In [7]:
daily_payload = daily_response["payload"]
daily_payload.get("Meta Data", {}) if isinstance(daily_payload, dict) else {}

{'1. Information': 'Daily Prices (open, high, low, close) and Volumes',
 '2. Symbol': 'IBM',
 '3. Last Refreshed': '2026-03-20',
 '4. Output Size': 'Compact',
 '5. Time Zone': 'US/Eastern'}

In [8]:
time_series_key = "Time Series (Daily)"
daily_series = daily_payload.get(time_series_key, {}) if isinstance(daily_payload, dict) else {}

daily_prices = pd.DataFrame.from_dict(daily_series, orient="index")
daily_prices = daily_prices.reset_index().rename(columns={"index": "price_date"})
daily_prices.columns = [
    column.replace("1. ", "")
    .replace("2. ", "")
    .replace("3. ", "")
    .replace("4. ", "")
    .replace("5. ", "")
    .replace(" ", "_")
    .lower()
    for column in daily_prices.columns
]

if not daily_prices.empty:
    daily_prices["symbol"] = stock_symbol
    daily_prices["price_date"] = pd.to_datetime(daily_prices["price_date"])
    numeric_columns = [column for column in daily_prices.columns if column not in ["symbol", "price_date"]]
    daily_prices[numeric_columns] = daily_prices[numeric_columns].apply(pd.to_numeric, errors="coerce")
    daily_prices = daily_prices.sort_values("price_date", ascending=False)

daily_prices.head()

,price_date,open,high,low,close,volume,symbol
0,2026-03-20,249.00,250.28,241.77,241.77,11314093,IBM
1,2026-03-19,249.43,252.19,248.25,250.37,4249458,IBM
2,2026-03-18,254.16,258.28,250.16,251.60,5177047,IBM
3,2026-03-17,250.51,256.39,250.00,256.11,5840139,IBM
4,2026-03-16,247.87,252.20,246.10,249.25,5674228,IBM


In [9]:
if not daily_prices.empty:
    print("Row count:", len(daily_prices))
    print("Date range:", daily_prices["price_date"].min(), "to", daily_prices["price_date"].max())
    print("Duplicate business key count:", daily_prices.duplicated(subset=["symbol", "price_date"]).sum())
    print("Null counts:")
    display(daily_prices.isna().sum())
else:
    print("No daily price rows returned. Check the payload message above.")

Row count: 100
Date range: 2025-10-27 00:00:00 to 2026-03-20 00:00:00
Duplicate business key count: 0
Null counts:


price_date    0
open          0
high          0
low           0
close         0
volume        0
symbol        0
dtype: int64

## Step 5. Explore company overview

The `OVERVIEW` dataset is useful for attributes that may later become dimensions or slowly changing reference data, such as:
- company name
- sector
- industry
- country
- market capitalization
- valuation ratios
- dividend information

In [10]:
time.sleep(12)
overview_response = call_alpha_vantage(
    function="OVERVIEW",
    symbol=stock_symbol,
)

overview_payload = overview_response["payload"]
inspect_alpha_payload(overview_payload)

pd.Series({
    "Symbol": overview_payload.get("Symbol") if isinstance(overview_payload, dict) else None,
    "Name": overview_payload.get("Name") if isinstance(overview_payload, dict) else None,
    "Sector": overview_payload.get("Sector") if isinstance(overview_payload, dict) else None,
    "Industry": overview_payload.get("Industry") if isinstance(overview_payload, dict) else None,
    "Country": overview_payload.get("Country") if isinstance(overview_payload, dict) else None,
    "Currency": overview_payload.get("Currency") if isinstance(overview_payload, dict) else None,
    "MarketCapitalization": overview_payload.get("MarketCapitalization") if isinstance(overview_payload, dict) else None,
    "PERatio": overview_payload.get("PERatio") if isinstance(overview_payload, dict) else None,
    "DividendYield": overview_payload.get("DividendYield") if isinstance(overview_payload, dict) else None,
    "52WeekHigh": overview_payload.get("52WeekHigh") if isinstance(overview_payload, dict) else None,
    "52WeekLow": overview_payload.get("52WeekLow") if isinstance(overview_payload, dict) else None,
})

Symbol                                              IBM
Name                    International Business Machines
Sector                                       TECHNOLOGY
Industry                INFORMATION TECHNOLOGY SERVICES
Country                                             USA
Currency                                            USD
MarketCapitalization                       226879177000
PERatio                                           21.68
DividendYield                                    0.0268
52WeekHigh                                       323.06
52WeekLow                                         209.3
dtype: object

## Step 6. Explore financial statements

The two most useful fundamental datasets to inspect first are:
- `INCOME_STATEMENT`
- `BALANCE_SHEET`

They are helpful because they support future gold datasets such as profitability, leverage, and trend metrics.

In [11]:
time.sleep(12)
income_statement_response = call_alpha_vantage(
    function="INCOME_STATEMENT",
    symbol=stock_symbol,
)

income_statement_payload = income_statement_response["payload"]
inspect_alpha_payload(income_statement_payload)
list(income_statement_payload.keys()) if isinstance(income_statement_payload, dict) else []

['symbol', 'annualReports', 'quarterlyReports']

In [12]:
quarterly_income = pd.DataFrame(income_statement_payload.get("quarterlyReports", [])) if isinstance(income_statement_payload, dict) else pd.DataFrame()
quarterly_income.head()

,fiscalDateEnding,reportedCurrency,grossProfit,totalRevenue,costOfRevenue,costofGoodsAndServicesSold,operatingIncome,sellingGeneralAndAdministrative,researchAndDevelopment,operatingExpenses,investmentIncomeNet,netInterestIncome,interestIncome,interestExpense,nonInterestIncome,otherNonOperatingIncome,depreciation,depreciationAndAmortization,incomeBeforeTax,incomeTaxExpense,interestAndDebtExpense,netIncomeFromContinuingOperations,comprehensiveIncomeNetOfTax,ebit,ebitda,netIncome
0,2025-12-31,USD,12119000000,19686000000,7567000000,7567000000,4165000000,5462000000,2187000000,7954000000,None,-346000000,132000000,478000000,None,None,None,1297000000,4144000000,-1435000000,None,5579000000,None,4622000000,5919000000,5600000000
1,2025-09-30,USD,9591000000,16331000000,6740000000,6740000000,2660000000,4478000000,2082000000,6931000000,None,-342000000,150000000,492000000,None,None,None,1283000000,2430000000,686000000,None,1744000000,None,2922000000,4205000000,1744000000
2,2025-06-30,USD,9977000000,16977000000,7001000000,7001000000,3086000000,4322000000,2097000000,6890000000,None,-338000000,172000000,510000000,None,None,None,1265000000,2597000000,404000000,None,2193000000,None,3109000000,4374000000,2194000000
3,2025-03-31,USD,8031000000,14541000000,6510000000,6510000000,1765000000,4023000000,1950000000,6266000000,None,-264000000,191000000,455000000,None,None,None,1177000000,1158000000,103000000,None,1054000000,None,1613000000,2790000000,1055000000
4,2024-12-31,USD,10439000000,17553000000,7114000000,7114000000,3901000000,4325000000,1967000000,6538000000,None,-274000000,150000000,424000000,None,None,None,1113000000,3306000000,379000000,None,2927000000,None,3730000000,4843000000,2914000000


In [13]:
time.sleep(12)
balance_sheet_response = call_alpha_vantage(
    function="BALANCE_SHEET",
    symbol=stock_symbol,
)

balance_sheet_payload = balance_sheet_response["payload"]
inspect_alpha_payload(balance_sheet_payload)
quarterly_balance = pd.DataFrame(balance_sheet_payload.get("quarterlyReports", [])) if isinstance(balance_sheet_payload, dict) else pd.DataFrame()
quarterly_balance.head()

,fiscalDateEnding,reportedCurrency,totalAssets,totalCurrentAssets,cashAndCashEquivalentsAtCarryingValue,cashAndShortTermInvestments,inventory,currentNetReceivables,totalNonCurrentAssets,propertyPlantEquipment,accumulatedDepreciationAmortizationPPE,intangibleAssets,intangibleAssetsExcludingGoodwill,goodwill,investments,longTermInvestments,shortTermInvestments,otherCurrentAssets,otherNonCurrentAssets,totalLiabilities,totalCurrentLiabilities,currentAccountsPayable,deferredRevenue,currentDebt,shortTermDebt,totalNonCurrentLiabilities,capitalLeaseObligations,longTermDebt,currentLongTermDebt,longTermDebtNoncurrent,shortLongTermDebtTotal,otherCurrentLiabilities,otherNonCurrentLiabilities,totalShareholderEquity,treasuryStock,retainedEarnings,commonStock,commonStockSharesOutstanding
0,2025-12-31,USD,151880000000,35860000000,13641000000,13641000000,1220000000,17639000000,116020000000,9028000000,None,11391000000,11391000000,67717000000,None,None,830000000,2530000000,None,119140000000,38658000000,4756000000,None,None,7224000000,80482000000,3347000000,54836000000,6424000000,None,67154000000,8230000000,1068000000,32648000000,None,155648000000,63318000000,952400000
1,2025-09-30,USD,146312000000,32740000000,11569000000,11569000000,1397000000,12607000000,113572000000,9074000000,None,11729000000,11729000000,67396000000,None,None,3286000000,3881000000,None,118322000000,35142000000,3867000000,None,None,8749000000,83180000000,3453000000,55174000000,7942000000,None,66569000000,6985000000,11762000000,27905000000,None,151581000000,62819000000,948900000
2,2025-06-30,USD,148585000000,34253000000,11943000000,11943000000,1251000000,12747000000,114332000000,9258000000,None,12254000000,12254000000,67506000000,None,None,3504000000,4808000000,None,120997000000,37726000000,3974000000,None,None,9765000000,83271000000,3555000000,55219000000,8945000000,None,67719000000,7284000000,11522000000,27509000000,None,151367000000,62392000000,946700000
3,2025-03-31,USD,145667000000,35336000000,11035000000,11035000000,1431000000,11882000000,110330000000,9065000000,None,12391000000,12391000000,66065000000,None,None,6430000000,4558000000,None,118715000000,35106000000,3585000000,None,None,7711000000,83609000000,3551000000,56371000000,6913000000,None,66835000000,7180000000,11105000000,26880000000,None,150703000000,61913000000,945400000
4,2024-12-31,USD,137175000000,34482000000,13947000000,13947000000,1289000000,14010000000,102694000000,8928000000,None,10661000000,10661000000,60706000000,None,None,644000000,4592000000,None,109782000000,33142000000,4032000000,None,None,5857000000,76640000000,3423000000,49884000000,5089000000,None,58396000000,7313000000,981000000,27307000000,None,151163000000,61380000000,942400000


## Step 7. Explore FX daily rates

FX adds a global market angle to your platform and is very useful for BI. A simple pair like `USD/BRL` already improves the story of the project.

In [14]:
time.sleep(12)
fx_daily_response = call_alpha_vantage(
    function="FX_DAILY",
    from_symbol=fx_from,
    to_symbol=fx_to,
    outputsize="compact",
)

fx_payload = fx_daily_response["payload"]
inspect_alpha_payload(fx_payload)
list(fx_payload.keys()) if isinstance(fx_payload, dict) else []

['Meta Data', 'Time Series FX (Daily)']

In [15]:
fx_series = fx_payload.get("Time Series FX (Daily)", {}) if isinstance(fx_payload, dict) else {}
fx_daily = pd.DataFrame.from_dict(fx_series, orient="index")
fx_daily = fx_daily.reset_index().rename(columns={"index": "fx_date"})
fx_daily.columns = [
    column.replace("1. ", "")
    .replace("2. ", "")
    .replace("3. ", "")
    .replace("4. ", "")
    .replace(" ", "_")
    .lower()
    for column in fx_daily.columns
]

if not fx_daily.empty:
    fx_daily["from_symbol"] = fx_from
    fx_daily["to_symbol"] = fx_to
    fx_daily["fx_date"] = pd.to_datetime(fx_daily["fx_date"])
    numeric_columns = [column for column in fx_daily.columns if column not in ["fx_date", "from_symbol", "to_symbol"]]
    fx_daily[numeric_columns] = fx_daily[numeric_columns].apply(pd.to_numeric, errors="coerce")
    fx_daily = fx_daily.sort_values("fx_date", ascending=False)

fx_daily.head()

,fx_date,open,high,low,close,from_symbol,to_symbol
0,2026-03-20,5.2199,5.3257,5.2179,5.3136,USD,BRL
1,2026-03-19,5.2677,5.3140,5.2013,5.2198,USD,BRL
2,2026-03-18,5.1932,5.2689,5.1818,5.2673,USD,BRL
3,2026-03-17,5.2307,5.2415,5.1763,5.1923,USD,BRL
4,2026-03-16,5.3232,5.3232,5.2248,5.2310,USD,BRL


## Step 8. Explore one macroeconomic indicator

Macro indicators are valuable because they help you tell a stronger analytical story in the gold layer. We will start with `REAL_GDP`.

In [16]:
time.sleep(12)
real_gdp_response = call_alpha_vantage(function="REAL_GDP")
real_gdp_payload = real_gdp_response["payload"]
inspect_alpha_payload(real_gdp_payload)
list(real_gdp_payload.keys()) if isinstance(real_gdp_payload, dict) else []

['name', 'interval', 'unit', 'data']

In [17]:
real_gdp = pd.DataFrame(real_gdp_payload.get("data", [])) if isinstance(real_gdp_payload, dict) else pd.DataFrame()

if not real_gdp.empty:
    real_gdp["date"] = pd.to_datetime(real_gdp["date"])
    real_gdp["value"] = pd.to_numeric(real_gdp["value"], errors="coerce")
    real_gdp = real_gdp.sort_values("date", ascending=False)

real_gdp.head()

,date,value
0,2025-01-01,23852.994
1,2024-01-01,23358.435
2,2023-01-01,22723.719
3,2022-01-01,22075.931
4,2021-01-01,21532.407


## Step 9. Free-plan notes

When using the free plan, keep these exploration habits:
- do not fire many cells quickly one after another
- wait between calls when needed
- always inspect `Information`, `Note`, and `Error Message` fields
- treat those messages as first-class bronze metadata in your ingestion logs

This is why the notebook includes `time.sleep(12)` before later requests.

## Step 10. Bronze layer design

Recommended bronze datasets from Alpha Vantage using the free plan:
- `stock_prices_daily`
- `company_overview`
- `income_statement`
- `balance_sheet`
- `fx_daily`
- `macro_real_gdp`

Recommended raw path layout:

`s3://financial-data/bronze/source=alpha_vantage/dataset=stock_prices_daily/symbol=IBM/ingestion_date=2026-03-23/file.json`

Important metadata to store with each bronze record:
- source
- dataset
- natural key such as symbol or currency pair
- ingestion timestamp
- request parameters
- request URL
- raw payload


In [ ]:
ingestion_ts = datetime.now(timezone.utc).isoformat()

bronze_stock_prices_payload = {
    "source": "alpha_vantage",
    "dataset": "stock_prices_daily",
    "symbol": stock_symbol,
    "ingestion_ts_utc": ingestion_ts,
    "request_params": {
        "function": "TIME_SERIES_DAILY",
        "symbol": stock_symbol,
        "outputsize": "compact",
    },
    "request_url": daily_response["request_url"],
    "raw_payload": daily_payload,
}

print(json.dumps(bronze_stock_prices_payload, indent=2, default=str)[:2500])

In [19]:
ingestion_date = datetime.now(timezone.utc).strftime("%Y-%m-%d")

stock_prices_path = Path(
    f"bronze/source=alpha_vantage/dataset=stock_prices_daily/symbol={stock_symbol}/ingestion_date={ingestion_date}/daily.json"
)
overview_path = Path(
    f"bronze/source=alpha_vantage/dataset=company_overview/symbol={stock_symbol}/ingestion_date={ingestion_date}/overview.json"
)
fx_path = Path(
    f"bronze/source=alpha_vantage/dataset=fx_daily/from_symbol={fx_from}/to_symbol={fx_to}/ingestion_date={ingestion_date}/fx_daily.json"
)

print(stock_prices_path.as_posix())
print(overview_path.as_posix())
print(fx_path.as_posix())

bronze/source=alpha_vantage/dataset=stock_prices_daily/symbol=IBM/ingestion_date=2026-03-23/daily.json
bronze/source=alpha_vantage/dataset=company_overview/symbol=IBM/ingestion_date=2026-03-23/overview.json
bronze/source=alpha_vantage/dataset=fx_daily/from_symbol=USD/to_symbol=BRL/ingestion_date=2026-03-23/fx_daily.json


## Step 11. What we learned from this notebook

Best free Alpha Vantage datasets for this project:
- daily stock prices for market facts
- company overview for descriptive reference data
- financial statements for future analytical metrics
- FX for global market coverage
- macro indicators for richer gold-layer analysis

Best use of Alpha Vantage in your architecture:
- bronze: land raw JSON payloads exactly as returned
- silver: normalize price, FX, and fundamentals into typed tabular datasets
- gold: combine with Yahoo Finance market coverage and analytical models

Important tradeoff:
- Yahoo Finance is your better free source for dividends and splits
- Alpha Vantage free endpoints are still excellent for fundamentals, FX, and macro data

Suggested next build step after this notebook:
- create a small Python ingestion module that saves these raw payloads locally using the same bronze partition pattern we plan to use in S3